# 手撕 Label Smoothing

## 背景
将 one-hot 标签平滑为 (1-α)·one_hot + α/K，防止模型过度自信。
对应 loss = -sum(smoothed_label * log_softmax(logits))。

## 考察点
- 标签平滑的公式与直觉
- 与普通 cross entropy 的区别
- α 的选择（通常 0.1）

In [ ]:
import torch
import torch.nn.functional as F

def label_smoothing_loss(logits, target, alpha=0.1):
    K = logits.size(-1)
    log_probs = F.log_softmax(logits, dim=-1)
    # smoothed: (1-α)*one_hot + α/K
    nll = F.nll_loss(log_probs, target, reduction='none')  # -log_prob[target]
    uniform = -log_probs.mean(dim=-1)  # -mean(log_prob)
    return ((1 - alpha) * nll + alpha * uniform).mean()

def smooth_labels(target, K, alpha=0.1):
    smoothed = torch.full((target.size(0), K), alpha / K)
    smoothed.scatter_(1, target.unsqueeze(1), 1 - alpha + alpha / K)
    return smoothed

In [ ]:
# 对比普通 CE 和 label smoothing CE
torch.manual_seed(42)
logits = torch.randn(4, 10)
target = torch.tensor([0, 3, 7, 9])
ce_loss = F.cross_entropy(logits, target)
ls_loss = label_smoothing_loss(logits, target, alpha=0.1)
# 验证 smoothed label
smoothed = smooth_labels(target, 10, alpha=0.1)
assert torch.allclose(smoothed.sum(dim=-1), torch.ones(4)), "平滑后概率和=1"
assert torch.allclose(smoothed[0, 0], torch.tensor(1 - 0.1 + 0.1/10)), "目标位=1-α+α/K"
print(f"普通 CE:  {ce_loss.item():.4f}")
print(f"Label Smoothing CE: {ls_loss.item():.4f}")
print(f"平滑标签[0]: {smoothed[0].tolist()}")
print("✅ 标签平滑验证通过")